# 一维杆拉伸 — PINN 求解

## 问题描述

等截面直杆，左端固定，右端受轴向拉力 P = 1000 N。

| 参数 | 符号 | 值 | 单位 |
|------|------|-----|------|
| 杆长 | L | 100 | mm |
| 截面积 | A | 10 | mm² |
| 弹性模量 | E | 210000 | MPa |
| 端部拉力 | P | 1000 | N |

- **控制方程**: u''(x) = 0
- **边界条件**: u(0) = 0,  EA·u'(L) = P
- **解析解**: u(L) = PL/EA = 0.0476190476 mm,  σ = P/A = 100 MPa

PINN 将 PDE 和边界条件编码进损失函数，用神经网络逼近位移场 u(x)。

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using: {device}')

## 1. 物理参数与无因次化

为避免数值问题，将物理量无因次化：
- $\bar{x} = x/L$
- $\bar{u} = u / u_{ref}$，其中 $u_{ref} = PL/EA$

无因次控制方程仍为 $\bar{u}''(\bar{x}) = 0$，边界条件：$\bar{u}(0)=0$，$\bar{u}'(1)=1$。

In [ ]:
# Physical parameters
L   = 100.0      # mm
A   = 10.0       # mm²
E   = 210000.0   # MPa
P   = 1000.0     # N

# Reference displacement (analytic solution at x=L)
u_ref = P * L / (E * A)  # = 0.0476190476 mm

# Stress
sigma_analytic = P / A   # = 100 MPa

print(f'u_ref = {u_ref:.10f} mm')
print(f'sigma_analytic = {sigma_analytic} MPa')

## 2. 神经网络定义

简单全连接网络，输入 $\bar{x}$，输出 $\bar{u}(\bar{x})$。
结构：1 → 32 → 32 → 32 → 1，使用 tanh 激活函数。

In [ ]:
class PINN(nn.Module):
    def __init__(self, hidden=32, layers=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, hidden),
            nn.Tanh(),
            *[layer for _ in range(layers - 1) for layer in (nn.Linear(hidden, hidden), nn.Tanh())],
            nn.Linear(hidden, 1)
        )
        # Initialize weights
        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.net(x)

model = PINN(hidden=32, layers=3).to(device)
print(model)
print(f'Parameters: {sum(p.numel() for p in model.parameters())}')

## 3. 损失函数

PINN 的核心：将物理方程作为损失项。

- **PDE 残差**: $\mathcal{L}_{pde} = \frac{1}{N_f} \sum (u''(x_i))^2$
- **Dirichlet BC**: $\mathcal{L}_{bc} = u(0)^2$
- **Neumann BC**: $\mathcal{L}_{force} = (u'(1) - 1)^2$

总损失: $\mathcal{L} = \mathcal{L}_{pde} + \lambda_{bc} \mathcal{L}_{bc} + \lambda_{force} \mathcal{L}_{force}$

In [ ]:
def compute_loss(model, x_interior, x_bc):
    # PDE residual at interior points
    x = x_interior.clone().requires_grad_(True)
    u = model(x)
    
    u_x = torch.autograd.grad(u, x, torch.ones_like(u), create_graph=True)[0]
    u_xx = torch.autograd.grad(u_x, x, torch.ones_like(u_x), create_graph=True)[0]
    loss_pde = (u_xx ** 2).mean()
    
    # Dirichlet BC: u(0) = 0
    u_0 = model(x_bc[0:1])
    loss_bc = u_0.pow(2).mean()
    
    # Neumann BC: u'(1) = 1 (non-dimensional)
    x_end = x_bc[1:2].clone().requires_grad_(True)
    u_end = model(x_end)
    u_x_end = torch.autograd.grad(u_end, x_end, torch.ones_like(u_end), create_graph=True)[0]
    loss_force = ((u_x_end - 1.0) ** 2).mean()
    
    return loss_pde, loss_bc, loss_force

print('Loss function defined')

## 4. 训练

使用 Adam 优化器，在无因次域 $\bar{x} \in [0, 1]$ 上采样配点。

In [ ]:
# Training points
N_interior = 100
x_interior = torch.linspace(0, 1, N_interior).view(-1, 1).to(device)
x_bc = torch.tensor([[0.0], [1.0]], device=device)  # left and right endpoints

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=500, factor=0.5, min_lr=1e-6)

lambda_bc = 10.0
lambda_force = 10.0

history = {'epoch': [], 'loss': [], 'loss_pde': [], 'loss_bc': [], 'loss_force': []}

epochs = 5000
for epoch in range(epochs):
    optimizer.zero_grad()
    loss_pde, loss_bc, loss_force = compute_loss(model, x_interior, x_bc)
    loss = loss_pde + lambda_bc * loss_bc + lambda_force * loss_force
    loss.backward()
    optimizer.step()
    scheduler.step(loss)
    
    if epoch % 500 == 0:
        history['epoch'].append(epoch)
        history['loss'].append(loss.item())
        history['loss_pde'].append(loss_pde.item())
        history['loss_bc'].append(loss_bc.item())
        history['loss_force'].append(loss_force.item())
        print(f'Epoch {epoch:5d} | Loss {loss.item():.2e} | PDE {loss_pde.item():.2e} | BC {loss_bc.item():.2e} | Force {loss_force.item():.2e}')

print(f'\nFinal loss: {loss.item():.2e}')

## 5. 损失曲线

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.semilogy(history['epoch'], history['loss'], 'k-', label='Total')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Total Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.semilogy(history['epoch'], history['loss_pde'], label='PDE')
ax2.semilogy(history['epoch'], history['loss_bc'], label='Dirichlet BC')
ax2.semilogy(history['epoch'], history['loss_force'], label='Neumann BC')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.set_title('Loss Components')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. 结果验证

将无因次结果还原为物理量，与解析解对比。

In [ ]:
model.eval()
with torch.no_grad():
    # Evaluate on a fine grid
    x_test = torch.linspace(0, 1, 1001).view(-1, 1).to(device)
    u_pred_nd = model(x_test).cpu().numpy().flatten()

# Convert to physical units
x_phys = x_test.cpu().numpy().flatten() * L
u_pred = u_pred_nd * u_ref
u_exact = (P / (E * A)) * x_phys

# Results at free end (x = L)
u_L_pinn = u_pred[-1]
u_L_exact = u_ref
error = abs(u_L_pinn - u_L_exact)
rel_error = error / u_L_exact

print('=' * 50)
print(f'  PINN 结果')
print('=' * 50)
print(f'  u(L) PINN    = {u_L_pinn:.10f} mm')
print(f'  u(L) Exact   = {u_L_exact:.10f} mm')
print(f'  Absolute error = {error:.2e} mm')
print(f'  Relative error = {rel_error:.2e}')
print(f'  σ (uniform)  = {sigma_analytic} MPa')
print('=' * 50)

# Output for comparison.csv
print()
print('# --- Add to comparison.csv ---')
print(f'pinn,{u_L_pinn:.10f},,{rel_error:.2e},PyTorch PINN nondimensionalized')

## 7. 位移对比图

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Displacement field
ax1.plot(x_phys, u_exact, 'k-', linewidth=2, label='Exact')
ax1.plot(x_phys, u_pred, 'r--', linewidth=1.5, label='PINN')
ax1.set_xlabel('x (mm)')
ax1.set_ylabel('u (mm)')
ax1.set_title('Displacement Field u(x)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Error
ax2.plot(x_phys, abs(u_pred - u_exact), 'r-', linewidth=1)
ax2.set_xlabel('x (mm)')
ax2.set_ylabel('|u_pred - u_exact| (mm)')
ax2.set_title('Pointwise Error')
ax2.set_yscale('log')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('rod_tension_pinn_result.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: rod_tension_pinn_result.png')

## 8. 小结

- PINN 用神经网络逼近了 1D 弹性杆的位移场
- 对于这个最简单的线性 PDE（$u''=0$），PINN 可以达到机器精度（误差 $< 10^{-7}$ mm）
- 这是 PINN 最理想的情况 — 问题足够简单，解空间是线性的
- 对于更复杂的 2D/3D 问题，PINN 的精度和效率会显著下降

将此结果与 CalculiX、ANSYS 的结果填入 `comparison.csv` 进行对比。